# Session 6: LLM을 활용한 임상 데이터 분석
**OpenRouter API + ZDR (Zero Data Retention) 적용 버전**

| Part | 내용 |
|------|------|
| **Part 0** | 환경 설정 (OpenRouter API, ZDR 설정 및 검증) |
| **Part 1** | Clinical Note 불러오기 |
| **Part 2** | Clinical Note Analysis |
|  | - Task 1: 약물명 추출 및 ATC 코드 식별 |
|  | - Task 2: 약물 부작용(ADR) 탐지 |
|  | - Task 3: 임상노트 요약 |
| **Part 3** | InBody 리포트 이미지 → 구조화된 데이터 변환 |

---
# Part 0. 환경 설정

## 0.1 패키지 설치 및 임포트

In [49]:
# 필요 패키지 설치 (최초 1회)
# pip install python-dotenv pandas requests

import pandas as pd
import requests
import json
import os
import base64
import re
from dotenv import load_dotenv

## 0.2 OpenRouter API 설정

[OpenRouter](https://openrouter.ai/)는 다양한 LLM 모델(OpenAI, Anthropic, Google 등)을 **하나의 API**로 호출할 수 있는 통합 게이트웨이입니다.

> `.env` 파일에 `OPENROUTER_API_KEY`를 저장한 뒤 불러옵니다.
> ```
> # .env 파일 예시
> OPENROUTER_API_KEY=sk-or-v1-xxxxxxxxxxxx
> ```

In [50]:
# .env 파일에서 API 키 로드
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# OpenRouter API 기본 설정
BASE_URL = "https://openrouter.ai/api/v1"

# 데이터 파일 경로 (본인의 경로로 수정하세요)
PATH = "/Users/moonie/Desktop/GCDA_2026_Tutorial/Session6"
os.makedirs(PATH, exist_ok=True)

# 사용할 모델 설정 (OpenRouter 모델명 형식: provider/model)
MODEL = "openai/gpt-5.4"            # 메인 모델
MODEL_MINI = "openai/gpt-5.4-mini"  # 경량 모델 (검증용)

print(f"API Base URL: {BASE_URL}")
print(f"Main Model:  {MODEL}")
print(f"Mini Model:  {MODEL_MINI}")

API Base URL: https://openrouter.ai/api/v1
Main Model:  openai/gpt-5.4
Mini Model:  openai/gpt-5.4-mini


## 0.3 ZDR (Zero Data Retention) 설정

**ZDR이란?**
- LLM API에 전송한 데이터가 모델 학습이나 서비스 개선에 사용되지 않도록 **데이터 보존을 거부**하는 설정입니다.
- 의료 데이터처럼 민감한 정보를 다룰 때 필수적으로 고려해야 합니다.

**OpenRouter에서의 ZDR 적용 방법**
- 요청 본문에 `provider.zdr: true`를 추가하면, **ZDR을 지원하는 엔드포인트로만 요청이 라우팅**됩니다.
- ZDR이 활성화되면 해당 provider는 입력/출력 데이터를 저장하지 않습니다.
- 참고: https://openrouter.ai/docs/guides/features/zdr

아래 `call_llm()` 함수는 모든 API 호출에 ZDR flag를 자동으로 적용합니다.

In [51]:
def call_llm(prompt, system_prompt="", model=MODEL, image_base64=None):
    """
    OpenRouter API 호출 함수 (ZDR 자동 적용)
    requests 라이브러리로 직접 호출하여 응답 호환성 문제를 방지합니다.

    Parameters:
        prompt (str): 사용자 프롬프트
        system_prompt (str): 시스템 프롬프트 (선택)
        model (str): 사용할 모델명 (기본: MODEL)
        image_base64 (str): base64 인코딩된 이미지 (선택, Vision 모델용)

    Returns:
        dict: API 응답 (JSON)
    """
    messages = []

    # System prompt 설정
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    # User message 구성 (텍스트 또는 텍스트+이미지)
    if image_base64:
        messages.append({
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
            ]
        })
    else:
        messages.append({"role": "user", "content": prompt})

    # API 호출 (ZDR flag 포함)
    resp = requests.post(
        f"{BASE_URL}/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": messages,
            "provider": {
                "zdr": True  # << ZDR: Zero Data Retention 활성화
            }
        }
    )

    result = resp.json()

    # 에러 체크
    if "error" in result:
        print(f"[API Error] {result['error']}")

    return result


def get_text(response):
    """응답에서 텍스트만 추출 (다양한 응답 구조 대응)"""
    try:
        return response["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        # choices가 없는 경우 - 전체 응답 출력하여 구조 확인
        print("[DEBUG] 'choices' 키를 찾을 수 없습니다. 전체 응답:")
        print(json.dumps(response, indent=2, ensure_ascii=False))
        return None


def get_usage(response):
    """응답에서 토큰 사용량 추출"""
    return response.get("usage", {})

## 0.4 ZDR 동작 확인

`requests` 라이브러리로 직접 API를 호출하여, ZDR flag가 포함된 요청이 정상적으로 처리되는지 확인합니다.

**확인 방법:**
1. 전송되는 요청 본문에 `provider.zdr: true` 가 포함되어 있는지 확인
2. 요청이 정상 처리(200)되는지 확인
3. OpenRouter의 Generation 상세 정보를 조회하여 실제 적용 여부 확인

In [52]:
payload = {
    "model": MODEL_MINI,
    "messages": [{"role": "user", "content": "Reply with only: ZDR_TEST_OK"}],
    "provider": {
        "zdr": True
    }
}

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    # 라우팅 메타데이터를 응답에 포함시키기 위한 옵트인 헤더
    "X-OpenRouter-Experimental-Metadata": "enabled",
}

print("=== 전송 요청 본문 ===")
print(json.dumps(payload, indent=2, ensure_ascii=False))
print()

resp = requests.post(
    f"{BASE_URL}/chat/completions",
    headers=headers,
    json=payload,
    timeout=60,
)

print(f"Status: {resp.status_code}")

if resp.status_code != 200:
    print(f"Error: {resp.text}")
    raise SystemExit(1)

data = resp.json()

print(f"Model: {data.get('model', 'N/A')}")
print("Response:", data["choices"][0]["message"]["content"])

# 1) 라우팅 메타데이터 확인
metadata = data.get("openrouter_metadata")
print("\n=== openrouter_metadata ===")
print(json.dumps(metadata, indent=2, ensure_ascii=False) if metadata is not None else "없음")

# 2) ZDR 가능한 엔드포인트 목록과 대조
zdr_resp = requests.get(
    f"{BASE_URL}/endpoints/zdr",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    },
    timeout=60,
)

print(f"\nZDR endpoint list status: {zdr_resp.status_code}")

if zdr_resp.status_code == 200:
    zdr_data = zdr_resp.json()
    endpoints = zdr_data.get("data", [])

    # 모델 ID가 응답 metadata나 payload model과 일치하는지 확인
    model_id = data.get("model") or payload["model"]
    matched = [
        ep for ep in endpoints
        if ep.get("model_id") == model_id or ep.get("model_id") == payload["model"]
    ]

    print(f"Model '{model_id}' is in ZDR list: {bool(matched)}")
    
    if matched:
        print("Matched endpoint(s):")
        print(json.dumps(matched, indent=2, ensure_ascii=False))
else:
    print(f"Error: {zdr_resp.text}")

=== 전송 요청 본문 ===
{
  "model": "openai/gpt-5.4-mini",
  "messages": [
    {
      "role": "user",
      "content": "Reply with only: ZDR_TEST_OK"
    }
  ],
  "provider": {
    "zdr": true
  }
}

Status: 200
Model: openai/gpt-5.4-mini-20260317
Response: ZDR_TEST_OK

=== openrouter_metadata ===
{
  "requested": "openai/gpt-5.4-mini",
  "strategy": "direct",
  "region": "ICN",
  "summary": "available=1, selected=Azure",
  "attempt": 1,
  "is_byok": false,
  "endpoints": {
    "total": 2,
    "available": [
      {
        "provider": "Azure",
        "model": "openai/gpt-5.4-mini-20260317",
        "selected": true
      }
    ]
  }
}

ZDR endpoint list status: 200
Model 'openai/gpt-5.4-mini-20260317' is in ZDR list: True
Matched endpoint(s):
[
  {
    "name": "Azure | openai/gpt-5.4-mini-20260317",
    "model_id": "openai/gpt-5.4-mini",
    "model_name": "OpenAI: GPT-5.4 Mini",
    "context_length": 400000,
    "pricing": {
      "prompt": "0.00000075",
      "completion": "0.0000045",
 

---
# Part 1. Clinical Note 불러오기

**1) Clinical Note 불러 오는 방법 안내**

파일에서 Clinical Note 불러오기

- 텍스트(.txt) 파일에 작성된 노트를 불러오는 방식입니다.
- 불러올 파일 경로를 정확히 지정해야 합니다.


**2) 예시로 사용되는 note1과 note2는 서로 다른 내용입니다:**
- note1: shorthand 스타일의 간단한 요약형 노트
- note2: narrative 스타일의 서술형 노트로, 보다 상세한 문맥과 진단 경과가 포함되어 있습니다

이 두 노트를 각각 사용하여 prompt 차이, 언어 차이, 노트 형식 차이에 따른 분석 결과 차이를 실험해볼 수 있습니다.

In [53]:
# note1 불러오기 (.txt 파일)
note1_path = os.path.join(PATH, "note1_ShorthandStyle.txt")
with open(note1_path, "r", encoding="utf-8") as f:
    note1 = f.read()

In [54]:
# note1 내용 확인
print(note1)

#. T2DM  
   Dx since 2010, recent HbA1c 9.1  
   MTF → 설사, 복부팽만, 식욕저하, 무기력감  
   Lactic acidosis (Lac 3.9) 의심 → 약 중단, 수액 치료  
   MTF-induced lactic acidosis 의심 → 약 중단 후 회복, basal insulin 예정

   Levemir 22 --- 10 units, 저녁 인슐린 잘 안 맞음


#. HTN  
   BP on admission 92/58
   Losartan 유지 

#. CKD stage 3  
   Baseline Cr 1.6 → 입원 시 Cr 2.1  
   수액 이후 신기능 호전, 투석 필요 없음  

#. Dyslipidemia  
   Atorvastatin  



### 1.2 Narrative version (MIMIC-style)

In [55]:
# note2 불러오기 (.txt 파일)
note2_path = os.path.join(PATH, "note2_MIMICstyle.txt")
with open(note2_path, "r", encoding="utf-8") as f:
    note2 = f.read()

In [56]:
# note2 앞 500자만 확인
print(note2[:500])

History of Present Illness:  
78-year-old female with a history of type 2 diabetes, hypertension, and CKD stage 3 who presented with generalized weakness, poor oral intake, and nausea.

Two weeks prior to admission, she was started on metformin after a primary care visit revealed HbA1c of 9.1. Since initiation, the patient has developed persistent loose stools and intermittent abdominal cramping. No reported fever or vomiting. Her family noted a gradual decline in appetite and energy over the pa


### 노트 선택

> **본 실습에서는 note1 (shorthand version) 을 사용합니다.**
> Narrative version을 사용하려면 아래 코드를 수정하세요.

In [57]:
# 사용할 노트 선택
# note = note2  # MIMIC-style narrative version
note = note1    # Shorthand version (기본값)

---
# Part 2. Clinical Note Analysis

## [Task 1] Drug Name Extraction and Drug Code Identification

이 Task에서는 임상노트로부터 언급된 약물명을 추출하는 작업을 수행합니다.

- GPT에게 임상노트를 입력하고, 텍스트에서 **약물명** 추출 및 약물에 상응하는 **ATC CODE**를 추출하게 합니다.
- 약물은 **일반명** 또는 **약어**(예: MFM = metformin)로 표현될 수 있습니다.

### (1) Drug Entity Recognition

> **Zero-shot prompting** 예시: 예시 없이 지시문만으로 약물명을 추출합니다.

In [58]:
# Zero-shot 프롬프트: 약물명만 추출
prompt_drug_name = f"""
Task: Extract all drug names mentioned in the following clinical note.
- Drug names may be written as abbreviations.
- Return the list of drug names only, without any additional explanation.

Clinical Note:
{note}
"""

In [59]:
# API 호출 및 결과 확인
response = call_llm(
    prompt=prompt_drug_name,
    system_prompt="You are a clinical text extraction assistant.",
    model=MODEL
)

print(get_text(response))

MTF
Levemir
Losartan
Atorvastatin


### (2) Drug Entity + Description + ATC Code

> **Few-shot prompting** 예시: 예시(example)를 함께 제공하여 출력 형식을 유도합니다.

In [61]:
prompt_wATC = f"""

## Task
Given a clinical note, extract:
1. Drug names clearly or implicitly mentioned (including abbreviations).
2. A short description of each drug (1 line).
3. The ATC code corresponding to the drug (ATC Level 5).

Output must be in CSV format with the columns:
Drug, Description, ATC_Code

## Examples:
Clinical Note 1:
"ASA 투여 후 환자 두드러기 호소. PRN으로 항히스타민제 처방."

Output (CSV):
Drug,Description,ATC_Code
ASA,Aspirin (NSAID used for pain/fever, antiplatelet),B01AC06
항히스타민제,Antihistamine for allergic symptoms,R06AE07

Clinical Note 2:
"ASA 투여 후 환자 두드러기 호소. PRN으로 항히스타민제 처방."

Output (CSV):
Drug,Description,ATC_Code
ASA,Aspirin (NSAID used for pain/fever, antiplatelet),B01AC06
항히스타민제,Antihistamine for allergic symptoms,R06AE07

Now process the user input below.
Clinical Note:
{note}
"""

In [62]:
# API 호출 및 결과 확인
response = call_llm(
    prompt=prompt_wATC,
    system_prompt="You are a clinical text extraction assistant specializing in drug detection from clinical notes.",
    model=MODEL
)

print(get_text(response))

Drug,Description,ATC_Code
MTF,Metformin (biguanide oral antidiabetic for type 2 diabetes),A10BA02
Levemir,Insulin detemir (long-acting basal insulin),A10AE05
Losartan,Angiotensin II receptor blocker used for hypertension,AII09CA01
Atorvastatin,HMG-CoA reductase inhibitor used for dyslipidemia,C10AA05


### (3) Token 사용량 확인

사용한 토큰을 아래의 코드를 이용해서 체크할 수 있습니다.

In [65]:
# 토큰 사용량 확인
usage = response.get("usage", {})
print("Total:", usage.get("total_tokens"))

Total: 556


## [Task 2] Adverse Drug Reaction Detection

이 Task에서는 임상노트에 언급된 **약물 부작용(Adverse Drug Reactions, ADR)** 을 탐지하는 작업을 수행합니다.

- GPT에게 임상노트를 입력하고, 특정 약물 투여 이후 발생한 부작용을 **약물-증상 쌍(drug → symptom)** 형태로 추출하게 합니다.
- 약물명은 약어로 표현될 수 있으며, 부작용은 명시적 또는 암시적으로 표현되어 있을 수 있습니다.

In [66]:
# ADR 추출 프롬프트
prompt_adr = f"""
Task: Extract all possible adverse drug reactions (ADRs) mentioned in the clinical note.
- Return results in JSON format, with each item containing:
  - "drug": drug name
  - "symptom": symptom or adverse effect
  - "evidence": supporting sentence or phrase from the note
- No extra explanation.

Clinical Note:
{note}
"""

In [67]:
# Extractor Agent 호출
response = call_llm(
    prompt=prompt_adr,
    system_prompt="You are a clinical text extraction assistant specializing in identifying adverse drug reactions (ADRs).",
    model=MODEL
)

extractor_output = get_text(response)
print(extractor_output)

[
  {
    "drug": "MTF",
    "symptom": "설사",
    "evidence": "MTF → 설사, 복부팽만, 식욕저하, 무기력감"
  },
  {
    "drug": "MTF",
    "symptom": "복부팽만",
    "evidence": "MTF → 설사, 복부팽만, 식욕저하, 무기력감"
  },
  {
    "drug": "MTF",
    "symptom": "식욕저하",
    "evidence": "MTF → 설사, 복부팽만, 식욕저하, 무기력감"
  },
  {
    "drug": "MTF",
    "symptom": "무기력감",
    "evidence": "MTF → 설사, 복부팽만, 식욕저하, 무기력감"
  },
  {
    "drug": "MTF",
    "symptom": "lactic acidosis",
    "evidence": "Lactic acidosis (Lac 3.9) 의심 → 약 중단, 수액 치료"
  },
  {
    "drug": "MTF",
    "symptom": "lactic acidosis",
    "evidence": "MTF-induced lactic acidosis 의심 → 약 중단 후 회복, basal insulin 예정"
  }
]


### (2) Verification Agent

위의 LLM이 생성한 답변이 맞는지 검증하기 위한 **별도의 Agent**를 생성해서 답변을 검증합니다.
Extractor → Verifier 구조의 **Multi-Agent 패턴**입니다.

In [68]:
# Verifier 프롬프트 구성
verifier_prompt = """
If you disagree, provide counterarguments.
Provide agree and comment for each drug and side effect pairs.
Respond in JSON:
{"agree": true/false, "comments": "..."}.
"""

verifier_input = f"{verifier_prompt}\n\nExtractorAgent output:\n{extractor_output}\n\nMedical note:\n{note}"

In [69]:
# Verifier Agent 호출 (경량 모델 사용)
verifier_response = call_llm(
    prompt=verifier_input,
    system_prompt="You are VerifierAgent. Your job is to check ExtractorAgent's identified side effects.",
    model=MODEL_MINI
)

print(get_text(verifier_response))

[
  {
    "agree": true,
    "comments": "설사, 복부팽만, 식욕저하, 무기력감은 노트에 명시된 MTF 관련 증상으로 타당합니다."
  },
  {
    "agree": true,
    "comments": "설사, 복부팽만, 식욕저하, 무기력감은 노트에 명시된 MTF 관련 증상으로 타당합니다."
  },
  {
    "agree": true,
    "comments": "설사, 복부팽만, 식욕저하, 무기력감은 노트에 명시된 MTF 관련 증상으로 타당합니다."
  },
  {
    "agree": true,
    "comments": "설사, 복부팽만, 식욕저하, 무기력감은 노트에 명시된 MTF 관련 증상으로 타당합니다."
  },
  {
    "agree": true,
    "comments": "Lactic acidosis (Lac 3.9) 의심, 약 중단 및 수액 치료, 그리고 MTF-induced lactic acidosis 의심 후 회복 내용이 있어 MTF와의 연관성은 적절합니다."
  },
  {
    "agree": true,
    "comments": "같은 근거를 다른 표현으로 반복한 항목이며, MTF-induced lactic acidosis 의심으로 판단하는 것이 타당합니다."
  }
]


## [Task 3] Clinical Note Summarization

이 Task에서는 임상노트에 포함된 핵심 정보를 GPT를 통해 한글로 **요약(summarization)** 하는 작업을 수행합니다.
- 임상노트에는 환자의 병력, 진단, 투약, 검사 결과, 치료 계획 등의 정보가 포함되어 있으며, 종종 매우 길고 복잡합니다.
- 본 Task의 목적은 다음 두 가지 유형의 요약을 생성하는 것입니다:
    - 환자/보호자용 요약
    - 의료진용 요약
- 출력은 한국어 요약문으로 생성됩니다.

### (1) 환자/보호자용 요약 (Family-Friendly Summary)

In [70]:
# 보호자용 요약 프롬프트
prompt_family = f"""
Task: Summarize the following clinical note in simple language for the patient's family.
- Use non-technical, plain Korean language.
- The summary should explain what happened, what was diagnosed, how it was treated, and what to expect.
- Avoid complex terminology and abbreviations.

Output language: Korean

Clinical Note:
{note}
"""

In [71]:
# 보호자용 요약 생성
response = call_llm(
    prompt=prompt_family,
    system_prompt="You are a clinical text summarization assistant.",
    model=MODEL
)

print(get_text(response))

가족분들께 설명드리면,

환자분은 **당뇨병, 혈압 문제, 신장 기능 저하, 콜레스테롤 문제**가 있는 상태입니다.

이번에는 **당뇨약 때문에 몸 상태가 나빠진 것으로 보였습니다.**  
기존에 먹던 당뇨약을 먹은 뒤에 **설사, 배가 더부룩함, 입맛 저하, 기운 없음**이 있었고, 검사에서 몸에 **젖산이 쌓였을 가능성**이 보여서 그 약은 **중단**했습니다.

입원 당시에는 **혈압이 조금 낮았고**, **신장 기능도 평소보다 더 나빠져 있었습니다.**  
그래서 **수액 치료**를 했고, 그 뒤 **몸 상태와 신장 기능이 좋아졌습니다.**  
현재는 **투석이 필요할 정도는 아닙니다.**

당뇨 조절 상태는 최근 검사에서 **좋지 않은 편**이어서, 앞으로는 먹는 약 대신 **기본 인슐린 치료**를 시작할 예정입니다.  
다만 지금까지는 **저녁 인슐린을 규칙적으로 잘 맞지 못한 점**이 있어, 앞으로는 **정해진 시간에 꾸준히 맞는 것이 중요**합니다.

정리하면:
- **무슨 일이 있었나:** 당뇨약 복용 후 설사, 복부 불편감, 식욕 저하, 기운 없음이 생김
- **무엇으로 판단했나:** 당뇨약 때문에 몸에 무리가 왔고, 신장 기능도 일시적으로 나빠진 상태로 판단
- **어떻게 치료했나:** 문제되는 당뇨약을 끊고 수액 치료를 함
- **현재 상태는 어떤가:** 약을 끊은 뒤 회복 중이며, 신장 기능도 좋아지고 있음
- **앞으로는:** 먹는 당뇨약 대신 인슐린 중심으로 혈당을 조절할 가능성이 큼

기존의 **혈압약은 계속 유지**하고, **콜레스테롤 약도 계속 복용**합니다.

앞으로는 **혈당 관리, 인슐린 규칙적 사용, 탈수 예방, 신장 기능 추적 검사**가 중요합니다.


### (2) 의료진용 요약 (Clinician Summary)

In [72]:
# 의료진용 요약 프롬프트
prompt_doctor = f"""
Task: Summarize the following clinical note for a clinician.
- Use concise, professional clinical language.
- Include key diagnostic findings, relevant history, differential considerations, treatments given, response to treatment, and recommended next steps.
- Maintain clinical terminology and abbreviations (e.g., BP, HR, WNL, SOB, ASA, MRI).
- Do not simplify medical terms.
- Keep the summary focused and structured.

Output language: Korean (clinician-level medical terminology)

Clinical Note:
{note}
"""

In [73]:
# 의료진용 요약 생성
response = call_llm(
    prompt=prompt_doctor,
    system_prompt="You are a clinical text summarization assistant.",
    model=MODEL
)

print(get_text(response))

**임상 요약**

### 1. T2DM
- 2010년 진단, 최근 **HbA1c 9.1%**로 혈당 조절 불량.
- **MTF 복용 후** 설사, 복부팽만, 식욕저하, 무기력감 발생.
- **Lac 3.9**로 **MTF-induced lactic acidosis** 의심되어 약제 중단 및 수액 치료 시행.
- 약 중단 후 임상적으로 회복 양상.
- 향후 **basal insulin 시작 예정**.
- 기존 **Levemir 22 units → 10 units**로 조정된 것으로 보이며, **저녁 인슐린 순응도 불량**.

### 2. HTN
- 입원 당시 **BP 92/58**로 저혈압 경향.
- **Losartan 유지** 중.

### 3. CKD stage 3 / AKI 고려
- 기저 **Cr 1.6**, 입원 시 **Cr 2.1**로 상승.
- 탈수 또는 약제 관련 신기능 악화 가능성 고려.
- 수액 치료 후 신기능 호전되었으며 **투석 필요 없음**.

### 4. Dyslipidemia
- **Atorvastatin** 복용 중.

### 감별/임상적 고려사항
- **MTF-associated lactic acidosis**가 가장 의심되며, CKD stage 3 및 저혈압/탈수 상태가 유발 인자로 작용했을 가능성.
- 신기능 악화는 **prerenal AKI on CKD** 가능성 우선 고려.

### 치료 및 반응
- **MTF 중단**
- **IV fluid 치료**
- 치료 후 전신 증상 및 신기능 호전, lactate 관련 임상 상태 회복

### 권고/다음 단계
- **MTF 재투여는 신중히 회피 고려**
- **Basal insulin regimen**으로 당뇨 치료 전환 및 용량 재평가 필요
- 인슐린 **복약/주사 순응도 개선 교육** 필요
- **Cr/eGFR, 전해질, lactate, 혈당 추적**
- 저혈압 재평가하며 **ARB 지속 적절성** 모니터링 필요


---
# Part 3. InBody 리포트 이미지 → 구조화된 데이터 변환

이 Task에서는 **비정형 데이터인 Inbody Report 이미지**를 분석하여, 필요한 측정값을 **구조화된 Excel 데이터 형태**로 **자동 정리**하는 작업을 수행합니다.
- 인바디 리포트는 체성분(체중, 체지방률, 골격근량 등), 기초대사량, 신체 균형 지표 등 다양한 정보를 이미지 형태로 포함하고 있어, 직접 수기로 정리하기 어렵고 시간이 많이 소요됩니다.
- 본 튜토리얼에서는 GPT 모델을 활용하여 인바디 이미지에서 **핵심 수치·측정값을 자동으로 추출**하고, 이를 엑셀(Excel) 형태의 구조화된 표 (tabular data)로 변환하는 방법을 다룹니다.

## 3.1 이미지 로드 및 인코딩

In [74]:
# 이미지 파일 로드 및 base64 인코딩
image_path = os.path.join(PATH, "inbody_sample.jpg")

with open(image_path, "rb") as f:
    img_bytes = f.read()
    img_base64 = base64.b64encode(img_bytes).decode("utf-8")

print(f"이미지 로드 완료: {image_path}")
print(f"Base64 인코딩 길이: {len(img_base64):,} chars")

이미지 로드 완료: /Users/moonie/Desktop/GCDA_2026_Tutorial/Session6/inbody_sample.jpg
Base64 인코딩 길이: 484,336 chars


## 3.2 GPT Vision으로 데이터 추출

In [75]:
# InBody 데이터 추출 프롬프트
prompt_inbody = """
Extract the following values from the InBody report image.
Return in the exact format below. If not visible, leave blank.

## Basic Information
1. ID:
2. Height:
3. Age:
4. Gender:
5. Test Date/Time:

## Key Measurements
1. Weight:
2. Skeletal Muscle Mass (SMM):
3. Body Fat Mass:
4. BMI:
5. Percent Body Fat (PBF):
"""

In [76]:
# Vision 모델로 InBody 이미지 분석 (ZDR 적용)
response = call_llm(
    prompt=prompt_inbody,
    system_prompt="You are an AI assistant that extracts structured measurement values from an InBody body composition analysis report image.",
    model=MODEL,
    image_base64=img_base64
)

inbody_text = get_text(response)
print(inbody_text)

## Basic Information
1. ID: Jane Doe
2. Height: 163cm
3. Age: 41
4. Gender: Female
5. Test Date/Time: 2017.03.08. 16:47

## Key Measurements
1. Weight: 66.4 kg
2. Skeletal Muscle Mass (SMM): 26.7 kg
3. Body Fat Mass: 18.1 kg
4. BMI: 25.0
5. Percent Body Fat (PBF): 27.2%


## 3.3 텍스트 → DataFrame 변환

In [77]:
def parse_inbody_text_to_row(text):
    """
    InBody 텍스트 블록을 받아서 {'항목명': 값} 딕셔너리로 파싱
    """
    row_data = {}
    lines = text.strip().split("\n")

    for line in lines:
        line = line.strip()

        # section header는 건너뛰기
        if line.startswith("##"):
            continue

        # key:value 패턴 매칭
        # (?:\d+\.\s*)? = "1. " 같은 번호가 있어도 무시
        # (.+?)          = : 앞에 오는 문자열을 key로
        # \s*(.*)        = : 뒤에 오는 내용을 value로
        match = re.match(r"(?:\d+\.\s*)?(.+?):\s*(.*)", line)

        if match:
            key = match.group(1).strip()
            value = match.group(2).strip()
            row_data[key] = value

    return row_data

In [78]:
# 파싱 및 DataFrame 생성
parsed_row = parse_inbody_text_to_row(inbody_text)
df = pd.DataFrame([parsed_row])
df.head()

,ID,Height,Age,Gender,Test Date/Time,Weight,Skeletal Muscle Mass (SMM),Body Fat Mass,BMI,Percent Body Fat (PBF)
0,Jane Doe,163cm,41,Female,2017.03.08. 16:47,66.4 kg,26.7 kg,18.1 kg,25.0,27.2%


## 3.4 CSV 저장

In [79]:
# CSV 파일로 저장
csv_path = os.path.join(PATH, "inbody_sample.csv")
df.to_csv(csv_path, index=False)
print(f"저장 완료: {csv_path}")

저장 완료: /Users/moonie/Desktop/GCDA_2026_Tutorial/Session6/inbody_sample.csv
